In [2]:
import pandas as pd
import json

In [3]:
# 1. Dosya yollarını tanımlayalım
PATH_RAW = '../data/raw/'

# 2. Verileri okuyalım
users = pd.read_csv(f'{PATH_RAW}users_data.csv')
cards = pd.read_csv(f'{PATH_RAW}cards_data.csv')
# İşlem verisi çok büyük, şimdilik ilk 100.000 satır analiz için yeterli
transactions = pd.read_csv(f'{PATH_RAW}transactions_data.csv', nrows=100000)

with open(f'{PATH_RAW}mcc_codes.json', 'r') as f:
    mcc_codes = json.load(f)

with open(f'{PATH_RAW}train_fraud_labels.json', 'r') as f:
    fraud_labels = json.load(f)

print(":white_check_mark: Tüm veriler başarıyla yüklendi!")

# 3. İlk kontroller
print(f"\nToplam Kullanıcı Sayısı: {len(users)}")
print(f"Toplam Kart Sayısı: {len(cards)}")
print(f"İnceleyeceğimiz İşlem Sayısı: {len(transactions)}")

# Tabloya bir göz atalım
transactions.head()

:white_check_mark: Tüm veriler başarıyla yüklendi!

Toplam Kullanıcı Sayısı: 2000
Toplam Kart Sayısı: 6146
İnceleyeceğimiz İşlem Sayısı: 100000


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [4]:
# 1. Amount sütununu sayıya çevirelim
# Önce '$' işaretini siliyoruz, sonra sayıya (float) çeviriyoruz
transactions['amount'] = transactions['amount'].str.replace('$', '').astype(float)

# 2. Date sütununu gerçek tarih formatına alalım
transactions['date'] = pd.to_datetime(transactions['date'])

# 3. Zip kodunu string'e çevirip .0 kısmından kurtulalım (Varsa)
transactions['zip'] = transactions['zip'].astype(str).str.replace('.0', '', regex=False)

# 4. Negatif tutarları inceleyelim (İade işlemleri olabilir)
# Şimdilik mutlak değerini alabiliriz veya iade olarak işaretleyebiliriz
transactions['is_return'] = transactions['amount'] < 0
transactions['amount'] = transactions['amount'].abs()

print(":white_check_mark: Temizlik tamamlandı!")
transactions.head()

:white_check_mark: Temizlik tamamlandı!


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,is_return
0,7475327,2010-01-01 00:01:00,1556,2972,77.00,Swipe Transaction,59935,Beulah,ND,58523,5499,NaN,True
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,5311,NaN,False
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,4829,NaN,False
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,4829,NaN,False
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,5813,NaN,False


In [5]:
print("Transactions Sütunları:", transactions.columns.tolist())
print("Users Sütunları:", users.columns.tolist())

Transactions Sütunları: ['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_return']
Users Sütunları: ['id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']


In [6]:
print(f"Transaction client_id tipi: {transactions['client_id'].dtype}")
print(f"User id tipi: {users['id'].dtype}")

Transaction client_id tipi: int64
User id tipi: int64


In [7]:
# Benzersiz ID'leri küme (set) olarak alalım
txn_clients = set(transactions['client_id'].unique())
user_ids = set(users['id'].unique())

# Kaç kullanıcı ortak?
common_users = txn_clients.intersection(user_ids)

print(f"Transactions içindeki benzersiz kullanıcı sayısı: {len(txn_clients)}")
print(f"Users tablosundaki benzersiz kullanıcı sayısı: {len(user_ids)}")
print(f"Ortak kullanıcı sayısı: {len(common_users)}")

# Eğer ortak kullanıcı sayısı, txn_clients sayısından azsa veri kaybı riskin vardır!

Transactions içindeki benzersiz kullanıcı sayısı: 1083
Users tablosundaki benzersiz kullanıcı sayısı: 2000
Ortak kullanıcı sayısı: 1083


In [8]:
# client_id ve id sütunlarının ilk 5 değerine bakarak formatları aynı mı gör
print(transactions['client_id'].head())
print(users['id'].head())

0    1556
1     561
2    1129
3     430
4     848
Name: client_id, dtype: int64
0     825
1    1746
2    1718
3     708
4    1164
Name: id, dtype: int64


In [9]:
# 1. Transactions ve Users Birleştirmesi
# Transactions'daki client_id ile Users'daki id'yi eşleştiriyoruz.
df_combined = pd.merge(
    transactions,
    users,
    left_on='client_id',
    right_on='id',
    how='left',
    suffixes=('', '_user')
)



# 2. Kontrol (id_user sütunu oluşmuş olmalı, onu silebiliriz çünkü client_id ile aynı)
df_combined = df_combined.drop('id_user', axis=1)

# 3. Fraud Etiketlerini Ekleme
# JSON içindeki anahtarlar genellikle string formatındadır, o yüzden str(x) ile güvene alıyoruz.
df_combined['is_fraud'] = df_combined.index.map(lambda x: fraud_labels.get(str(x), 0))

print(f"Birleştirme sonrası yeni tablo boyutu: {df_combined.shape}")
print(f"Toplam dolandırıcılık vakası sayısı: {df_combined['is_fraud'].sum()}")

df_combined.head()

Birleştirme sonrası yeni tablo boyutu: (100000, 27)
Toplam dolandırıcılık vakası sayısı: 0


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,is_fraud
0,7475327,2010-01-01 00:01:00,1556,2972,77.00,Swipe Transaction,59935,Beulah,ND,58523,...,Female,594 Mountain View Street,46.80,-100.76,$23679,$48277,$110153,740,4,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,...,Male,604 Pine Street,40.80,-91.12,$18076,$36853,$112139,834,5,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,...,Male,2379 Forest Lane,33.18,-117.29,$16894,$34449,$36540,686,3,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,...,Female,903 Hill Boulevard,41.42,-87.35,$26168,$53350,$128676,685,5,0
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,...,Male,166 River Drive,38.86,-76.60,$33529,$68362,$96182,711,2,0
